In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

In [ ]:
df_events = pd.read_csv('earnings_transcripts_sorted.csv')
df_market = pd.read_csv('marketdata_to_events.csv')

In [ ]:
df_events.shape, df_market.shape

In [ ]:
df_events.head(), df_market.head()

## Market features within 30 trading days window prior to the event

In [ ]:
def build_market_features(market_df):

    features = []

    for event_id, event in market_df.groupby("event_id"):

        event = event.sort_values("Date").copy()
        pre = event[event["window"] == "pre"].copy()
        post = event[event["window"] == "post"].copy()

        # Pre-event features within the 30-trading-day window
        # Pre-event volatility
        pre["daily_return"] = pre["Close"].pct_change()
        pre_volatility = pre["daily_return"].std()

        # Pre-event momentum
        pre_momentum = (pre["Close"].iloc[-1] / pre["Close"].iloc[0] - 1)

        # Pre-event average volume
        pre_avg_volume = pre["Volume"].mean()

        # Earnings gap
        earnings_gap = np.nan

        if len(pre) > 0 and len(post) > 0:

            # Get the earnings announcement time
            timestamp = pd.Timestamp(event["earnings_timestamp"].iloc[0])

            event_time = timestamp.time()
            market_open = pd.Timestamp("09:30").time()
            market_close = pd.Timestamp("16:00").time()

            # Before-open events
            if event_time < market_open:
                previous_day = pre.iloc[-1]
                event_day = post.iloc[0]
                earnings_gap = (event_day["Open"] / previous_day["Close"] - 1)

            # During market hours events
            elif event_time < market_close:
                earnings_gap = np.nan

            # After-close events
            else:

                event_day = pre.iloc[-1]
                next_day = post.iloc[0]
                earnings_gap = (next_day["Open"] / event_day["Close"] - 1)

        features.append({
            "event_id": event_id,
            "symbol": event["symbol"].iloc[0],
            "pre_volatility": pre_volatility,
            "pre_momentum": pre_momentum,
            "pre_avg_volume": pre_avg_volume,
            "earnings_gap": earnings_gap
        })

    return pd.DataFrame(features)

In [ ]:
feature_mkt = build_market_features(df_market)
feature_mkt.head(),feature_mkt.shape, feature_mkt.isna().any(axis=1).sum()

## Traget variables for model pridiction

In [ ]:
def build_event_labels(market_df, horizons=(3, 7)):

    labels = []

    for event_id, event in market_df.groupby("event_id"):

        event = event.sort_values("Date")
        pre = event[event["window"] == "pre"]
        post = event[event["window"] == "post"].sort_values("Date")

        baseline_close = pre["Close"].iloc[-1]

        row = {
            "event_id": event_id,
            "symbol": event["symbol"].iloc[0]
        }

        for h in horizons:
            # First h post-announcement trading sessions
            post_h = post.head(h)
            ending_close = post_h["Close"].iloc[-1]

            post_return = ending_close / baseline_close - 1

            row[f"return_{h}d"] = post_return
            row[f"label_{h}d"] = int(post_return > 0)

        labels.append(row)

    return pd.DataFrame(labels)

In [ ]:
labels = build_event_labels(df_market)
labels.head(), labels.shape

In [ ]:
# df_model = (
#     df_clean
#     .merge(market_features_df, on=["event_id", "symbol"], how="inner")
#     .merge(event_labels_df, on=["event_id", "symbol"], how="inner")
# )

## Load FinBERT model

In [ ]:
model_name = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = torch.device("mps" if torch.mps.is_available() else "cpu")
model = model.to(device)
model.eval()

## Transcripts chunks and embeddings

In [ ]:
def embed_dataset(texts, chunk_size=450, batch_size=32):
    """
    Process the "content" column of the whole event dataset
    Generate one 768-D embedding per transcript

    Each earnings-call ranscript is split into chunks 
    FinBERT produces one CLS embedding per chunk, and the chunk embeddings are mean-pooled to create one embedding per transcript.

    Chunks from different transcripts are processed in the same batch.
    """

    model.eval()

    n_events = len(texts)
    hidden_size = model.config.hidden_size

    # Accumulate chunk embeddings for each event
    embedding_sums = np.zeros((n_events, hidden_size), dtype=np.float32)
    chunk_counts = np.zeros(n_events, dtype=np.int32)

    # Current batch info.
    batch_input_ids = []
    batch_event_ids = []
    batch_is_last_chunk = []

    progress = tqdm(total=n_events, desc="Embedding transcripts")

    def process_batch():
        """Run the current batch of chunks through FinBERT."""
        nonlocal batch_input_ids, batch_event_ids, batch_is_last_chunk

        if not batch_input_ids:
            return

        encoded = tokenizer.pad(
            [{"input_ids": ids} for ids in batch_input_ids],
            padding=True,
            return_tensors="pt"
        )

        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        with torch.inference_mode():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        # Get one CLS embedding per chunk: (batch_size, 768)
        cls_embeddings = outputs.last_hidden_state[:, 0, :].float()
        cls_embeddings = cls_embeddings.cpu().numpy()

        # Accumulate each chunk embedding into its corresponding event
        for j, event_idx in enumerate(batch_event_ids):
            embedding_sums[event_idx] += cls_embeddings[j]
            chunk_counts[event_idx] += 1

        # Update progress for transcripts whose final chunk was processed
        completed = sum(batch_is_last_chunk)
        progress.update(completed)

        # Clear batch
        batch_input_ids = []
        batch_event_ids = []
        batch_is_last_chunk = []

    # Iterate over the whole transcript dataset, tokenize the transcripts and chunk by 450 tokens
    for event_idx, text in enumerate(texts):

        # Handle empty or invalid transcripts
        if not isinstance(text, str) or not text.strip():
            progress.update(1)
            continue

        # Tokenize without truncating the full transcript
        token_ids = tokenizer(text, add_special_tokens=False, truncation=False)["input_ids"]

        # Split into chunks
        chunks = [
            token_ids[i:i + chunk_size]
            for i in range(0, len(token_ids), chunk_size)
        ]

        # Add special tokens and add chunks to the global batch
        for chunk_idx, chunk in enumerate(chunks):
            input_ids = ([tokenizer.cls_token_id] + chunk + [tokenizer.sep_token_id])

            batch_input_ids.append(input_ids) # (batch_size, 452)
            batch_event_ids.append(event_idx) # (batch_size, )
            batch_is_last_chunk.append(chunk_idx == len(chunks) - 1) # (batch_size, )

            # Process once the batch is full
            if len(batch_input_ids) >= batch_size:
                process_batch()

    # Process any remaining chunks
    process_batch()
    progress.close()

    # Mean-pool chunk embeddings for each event
    embeddings = np.full((n_events, hidden_size), np.nan, dtype=np.float32)

    valid = chunk_counts > 0
    embeddings[valid] = (embedding_sums[valid] / chunk_counts[valid, None])

    return embeddings

In [ ]:
embeddings = embed_dataset(
    df_events["content"].tolist(),
    chunk_size=450,
    batch_size=64
)

print(embeddings.shape)
# (number_of_events, 768)

## df_clean["embedding"] = list(embeddings)

In [ ]:
np.save("data/finbert_embeddings.npy", embeddings)

In [ ]:
emb_test = embed_dataset(df_events["content"][2345:2350], chunk_size = 450, batch_size=64)
emb_test, embeddings[2345:2350]

In [ ]:
df_events["event_id"]

In [ ]:
labels["event_id"]

## Construct full dataset with event and corresponding features and labels

In [ ]:
embdngs = np.load("data/finbert_embeddings.npy")
embdngs_df = pd.DataFrame(embdngs, columns=[f"embedding_{i}" for i in range(embdngs.shape[1])])
embdngs_df["event_id"] = df_events["event_id"].values
embdngs_df.head(), embdngs_df.shape

In [ ]:
df_model = (
    df_events[["event_id", "symbol", "date"]]
    .merge(
        feature_mkt,
        on=["event_id", "symbol"],
        how="inner"
    )
    .merge(
        labels,
        on=["event_id", "symbol"],
        how="inner"
    )
    .merge(
        embdngs_df,
        on="event_id",
        how="inner"
    )
)

In [ ]:
df_model.head(), df_model.shape

In [ ]:
df_model.to_csv("data/full_dataset_b4drp.csv", index=False)

In [ ]:
df_model_drop = df_model.dropna(axis=0, how='any')
df_model_drop.head(), df_model_drop.shape

In [ ]:
df_model_drop.to_csv("data/full_dataset.csv", index=False)